# 🛍️ Retail Sales Forecasting & Inventory Optimization
## Interactive Jupyter Notebook
---
This notebook walks through the complete pipeline interactively.
Run each cell in order.

**Modules:**
1. Dataset Generation
2. Data Quality & Preprocessing
3. Exploratory Data Analysis
4. Feature Engineering
5. Sales Forecasting (Random Forest)
6. Inventory Optimization
7. Business Insights


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='darkgrid')
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi']     = 100

print('✅ Imports successful')

## 1. Dataset Generation

In [ ]:
from generate_dataset import generate
import os

raw_path = '../data/retail_sales_data.csv'
if os.path.exists(raw_path):
    df_raw = pd.read_csv(raw_path, parse_dates=['date'])
    print(f'Loaded existing dataset: {df_raw.shape}')
else:
    print('Generating dataset...')
    df_raw = generate()
    df_raw.to_csv(raw_path, index=False)
    print(f'Generated: {df_raw.shape}')

df_raw.head()

In [ ]:
# Dataset overview
print('Dataset Shape:', df_raw.shape)
print('\nDate range:', df_raw.date.min().date(), '→', df_raw.date.max().date())
print('Stores:', df_raw.store.unique())
print('Categories:', df_raw.category.unique())
print('Products:', df_raw.product.nunique(), 'unique')
print('\nColumn types:')
print(df_raw.dtypes)

## 2. Data Quality & Preprocessing

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
print('Missing values:')
print(missing[missing > 0] if missing.sum() > 0 else 'None – dataset is clean ✅')
print('\nDuplicates:', df_raw.duplicated().sum())

In [ ]:
# Load clean dataset (generated by main.py preprocess step)
clean_path = '../data/retail_sales_clean.csv'
if os.path.exists(clean_path):
    df = pd.read_csv(clean_path, parse_dates=['date'])
    print(f'Clean dataset: {df.shape}')
else:
    from preprocess import preprocess_pipeline
    df = preprocess_pipeline()

print('New features added:')
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(new_cols)

## 3. Exploratory Data Analysis

In [ ]:
# Monthly revenue trend
monthly = df.groupby(df['date'].dt.to_period('M'))['revenue'].sum().reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()

fig, ax = plt.subplots()
ax.plot(monthly['date'], monthly['revenue'], color='#00b4d8', lw=2.5, marker='o', ms=3)
ax.fill_between(monthly['date'], monthly['revenue'], alpha=0.15, color='#00b4d8')
ax.set_title('Monthly Total Revenue (All Stores)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
plt.tight_layout(); plt.show()

In [ ]:
# Revenue by category
cat_rev = df.groupby('category')['revenue'].sum().sort_values()

fig, ax = plt.subplots(figsize=(10,5))
cat_rev.plot(kind='barh', ax=ax, color=sns.color_palette('Set2', len(cat_rev)))
ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
plt.tight_layout(); plt.show()

print('\nCategory revenue breakdown:')
print(cat_rev.sort_values(ascending=False).apply(lambda x: f'₹{x/1e6:.2f}M'))

In [ ]:
# Seasonal heatmap
df2 = df.copy()
df2['year']  = df2['date'].dt.year
df2['month'] = df2['date'].dt.month
pivot = df2.groupby(['year','month'])['units_sold'].sum().reset_index()
pivot = pivot.pivot(index='year', columns='month', values='units_sold')

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig, ax = plt.subplots(figsize=(14,4))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt=',', ax=ax, xticklabels=month_labels)
ax.set_title('Units Sold – Seasonal Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Forecasting Model

In [ ]:
FEATURES = [
    'month','day','day_of_week','week_of_year','quarter',
    'is_weekend','is_month_end',
    'lag_7','lag_14','lag_30',
    'rolling_mean_7','rolling_mean_30','rolling_std_7',
    'unit_price','discount_pct','promo_event',
    'stock_level','reorder_point',
    'store_enc','category_enc','product_enc',
]
TARGET = 'units_sold'

# Encode categoricals
df_enc = df.copy()
for col, enc_col in [('store','store_enc'),('category','category_enc'),('product','product_enc')]:
    le = LabelEncoder()
    df_enc[enc_col] = le.fit_transform(df_enc[col])

# Temporal split
cutoff = df_enc['date'].max() - pd.Timedelta(days=60)
train  = df_enc[df_enc['date'] <= cutoff]
test   = df_enc[df_enc['date'] >  cutoff]
print(f'Train: {len(train):,} | Test: {len(test):,}')

In [ ]:
# Train model
X_train, y_train = train[FEATURES].fillna(0), train[TARGET]
X_test,  y_test  = test[FEATURES].fillna(0),  test[TARGET]

model = RandomForestRegressor(n_estimators=100, max_depth=12,
                               min_samples_leaf=5, n_jobs=-1, random_state=42)
print('Training...')
model.fit(X_train, y_train)

preds = np.maximum(model.predict(X_test), 0)
mae   = mean_absolute_error(y_test, preds)
rmse  = np.sqrt(mean_squared_error(y_test, preds))
r2    = r2_score(y_test, preds)
mape  = np.mean(np.abs((y_test - preds)/(y_test+1))) * 100

print(f'\n── Model Results ────────────────')
print(f'  MAE  : {mae:.2f} units')
print(f'  RMSE : {rmse:.2f} units')
print(f'  R²   : {r2:.4f}')
print(f'  MAPE : {mape:.2f}%')

In [ ]:
# Actual vs Predicted plot
test_copy = test.copy()
test_copy['predicted'] = np.round(preds).astype(int)
daily = test_copy.groupby('date').agg(Actual=('units_sold','sum'), Predicted=('predicted','sum')).reset_index()

fig, ax = plt.subplots()
ax.plot(daily['date'], daily['Actual'],    label='Actual',    color='#e63946', lw=2)
ax.plot(daily['date'], daily['Predicted'], label='Predicted', color='#2a9d8f', lw=2, ls='--')
ax.fill_between(daily['date'], daily['Actual'], daily['Predicted'], alpha=0.1, color='orange')
ax.set_title('Actual vs Predicted – Daily Units Sold (Test Period)', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(10,6))
fi.plot(kind='barh', ax=ax, color='#4cc9f0')
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

## 5. Inventory Optimization

In [ ]:
LEAD_TIME = 7
Z_SCORE   = 1.65   # 95% service level
ORDER_COST = 50    # ₹ per order
HOLD_PCT   = 0.25  # 25% of unit price per year

hist = df.groupby(['store','product','category']).agg(
    mean_demand   = ('units_sold','mean'),
    std_demand    = ('units_sold','std'),
    current_stock = ('stock_level','last'),
    unit_price    = ('unit_price','first'),
    reorder_qty   = ('reorder_qty','first'),
).reset_index()
hist['std_demand'] = hist['std_demand'].fillna(0)

# Safety Stock
hist['safety_stock']  = (Z_SCORE * hist['std_demand'] * np.sqrt(LEAD_TIME)).round(1)

# Reorder Point
hist['reorder_point'] = (hist['mean_demand'] * LEAD_TIME + hist['safety_stock']).round(1)

# EOQ
def eoq(row):
    H = row['unit_price'] * HOLD_PCT
    D = row['mean_demand'] * 365
    if H <= 0: return 0
    return int(np.sqrt(2 * D * ORDER_COST / H))

hist['eoq']          = hist.apply(eoq, axis=1)
hist['days_of_stock'] = (hist['current_stock'] / (hist['mean_demand']+0.01)).round(1)

# Alert
def alert(row):
    if row['current_stock'] <= row['safety_stock']:   return '🔴 CRITICAL'
    if row['current_stock'] <= row['reorder_point']:  return '🟠 REORDER NOW'
    if row['days_of_stock'] < 14:                     return '🟡 LOW STOCK'
    return '🟢 OK'

hist['alert'] = hist.apply(alert, axis=1)

print('Inventory table sample:')
hist[['store','product','current_stock','safety_stock','reorder_point','eoq','days_of_stock','alert']].head(10)

In [ ]:
# Alert summary
print('── Alert Summary ─────────────────────────────')
print(hist['alert'].value_counts().to_string())
print()
print('Critical / Reorder items:')
critical = hist[hist['alert'].str.contains('CRITICAL|REORDER')]
print(critical[['store','product','category','current_stock','reorder_point','days_of_stock','alert']].head(15).to_string())

In [ ]:
# Stock health chart
sample = hist[hist['alert'].isin(['🔴 CRITICAL','🟠 REORDER NOW'])].head(15).sort_values('days_of_stock')
if len(sample):
    labels = sample['product'] + '\n(' + sample['store'] + ')'
    colors = ['#e63946' if 'CRITICAL' in a else '#f4a261' for a in sample['alert']]
    fig, ax = plt.subplots(figsize=(12,6))
    ax.barh(labels, sample['days_of_stock'], color=colors)
    ax.axvline(LEAD_TIME, color='blue', ls='--', lw=1.5, label=f'Lead Time ({LEAD_TIME}d)')
    ax.set_title('Stock Health – Days Remaining (Critical Items)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Days of Stock'); ax.legend()
    plt.tight_layout(); plt.show()

## 6. Business Insights Summary

In [ ]:
print('='*55)
print('      BUSINESS INSIGHTS SUMMARY')
print('='*55)

total_rev = df['revenue'].sum()
total_stockout_days = df['stockout'].sum()
total_lost_sales = df['lost_sales'].sum() * df['sell_price'].mean()
best_cat = df.groupby('category')['revenue'].sum().idxmax()
best_store = df.groupby('store')['revenue'].sum().idxmax()
peak_month = df.groupby('month')['revenue'].sum().idxmax()

print(f'  Total Revenue (2yr)     : ₹{total_rev/1e7:.2f} Crore')
print(f'  Stockout Incidents      : {total_stockout_days:,}')
print(f'  Estimated Lost Revenue  : ₹{total_lost_sales/1e5:.1f} Lakh')
print(f'  Best Category           : {best_cat}')
print(f'  Best Performing Store   : {best_store}')
print(f'  Peak Sales Month        : Month {peak_month}')
print(f'  Products Needing Reorder: {len(critical)}')
print('='*55)